In [2]:
import math


def alpha_beta_search(values, depth, branching):
    node_vals = {}  # Stores computed value for every visited (depth, index)
    pruned = set()  # Stores (depth, index) of pruned nodes

    def max_value(d, idx, alpha, beta):
        if d == depth:
            node_vals[(d, idx)] = values[idx]
            return values[idx]

        v = -math.inf
        for i in range(branching):
            child_idx = idx * branching + i
            v = max(v, min_value(d + 1, child_idx, alpha, beta))
            if v >= beta:
                for rem in range(i + 1, branching):
                    mark_pruned(d + 1, idx * branching + rem)
                node_vals[(d, idx)] = v
                return v
            alpha = max(alpha, v)
        node_vals[(d, idx)] = v
        return v

    def min_value(d, idx, alpha, beta):
        if d == depth:
            node_vals[(d, idx)] = values[idx]
            return values[idx]

        v = math.inf
        for i in range(branching):
            child_idx = idx * branching + i
            v = min(v, max_value(d + 1, child_idx, alpha, beta))
            if v <= alpha:
                for rem in range(i + 1, branching):
                    mark_pruned(d + 1, idx * branching + rem)
                node_vals[(d, idx)] = v
                return v
            beta = min(beta, v)
        node_vals[(d, idx)] = v
        return v

    def mark_pruned(d, idx):
        pruned.add((d, idx))
        if d < depth:
            for i in range(branching):
                mark_pruned(d + 1, idx * branching + i)

    optimal_val = max_value(0, 0, -math.inf, math.inf)
    return optimal_val, node_vals, pruned


def draw_tree(depth, branching, values, node_vals, pruned):
    slot_width = 8
    total_leaves = branching**depth
    width = total_leaves * slot_width

    def get_center(d, idx):
        span = width // (branching**d)
        return idx * span + span // 2

    def place_text(row, center, text):
        start = max(0, center - len(text) // 2)
        for k, ch in enumerate(text):
            if start + k < len(row):
                row[start + k] = ch

    print("\n--- Alpha-Beta Game Tree ---")
    for d in range(depth + 1):
        role = "LEAF" if d == depth else ("MAX " if d % 2 == 0 else "MIN ")
        node_row = [" "] * width

        for idx in range(branching**d):
            c = get_center(d, idx)
            if (d, idx) in pruned:
                label = f"{values[idx]}(X)" if d == depth else "X"
            else:
                label = str(node_vals[(d, idx)])
            place_text(node_row, c, label)

        print(f"{role} | " + "".join(node_row).rstrip())

        # Draw branch connectors to the next level
        if d < depth:
            stem_row = [" "] * width
            bar_row = [" "] * width
            drop_row = [" "] * width

            for idx in range(branching**d):
                p_center = get_center(d, idx)
                c_centers = [
                    get_center(d + 1, idx * branching + i)
                    for i in range(branching)
                ]

                stem_row[p_center] = "│"
                for x in range(c_centers[0], c_centers[-1] + 1):
                    bar_row[x] = "─"
                bar_row[c_centers[0]] = "┌"
                bar_row[c_centers[-1]] = "┐"
                bar_row[p_center] = "┼" if p_center in c_centers else "┴"

                for cc in c_centers:
                    drop_row[cc] = "│"

            print("     | " + "".join(stem_row).rstrip())
            print("     | " + "".join(bar_row).rstrip())
            print("     | " + "".join(drop_row).rstrip())


# --- Standard User Input ---
depth = int(input("Enter depth of the tree: "))
branching = int(input("Enter branching factor: "))
total_leaves = branching**depth

print(f"Enter {total_leaves} leaf node values separated by space:")
values = list(map(int, input().split()))

optimal_val, node_vals, pruned = alpha_beta_search(values, depth, branching)
draw_tree(depth, branching, values, node_vals, pruned)
print("\nOptimal Value:", optimal_val)
`

Enter depth of the tree: 3
Enter branching factor: 2
Enter 8 leaf node values separated by space:
10 9 14 18 5 4 50 3

--- Alpha-Beta Game Tree ---
MAX  |                                10
     |                                 │
     |                 ┌───────────────┴───────────────┐
     |                 │                               │
MIN  |                10                               5
     |                 │                               │
     |         ┌───────┴───────┐               ┌───────┴───────┐
     |         │               │               │               │
MAX  |        10              14               5               X
     |         │               │               │               │
     |     ┌───┴───┐       ┌───┴───┐       ┌───┴───┐       ┌───┴───┐
     |     │       │       │       │       │       │       │       │
LEAF |    10       9      14     18(X)     5       4     50(X)   3(X)

Optimal Value: 10
